In [ ]:
from neuron import h
from prev_ob_models.Birgiolas2020.isolated_cells import MC1, GC1
import numpy as np
import matplotlib.pyplot as plt

h.load_file("stdrun.hoc")

# test
print("NEURON version:", h.nrnversion())

### Cleaned parameter block

I consolidated the scattered simulation parameters into a single `params` dictionary to reduce repetition and make it easier to change settings. For backward compatibility, the code below also exports the common names (tstop, mc_input_delay, mc_input_amp, etc.) as top-level variables so the rest of the notebook continues to work unchanged.

Edit the values inside `params` to change the run configuration.

In [ ]:
# Consolidated parameters
params = {
    # Simulation controls
    "tstop": 6000.0,

    # External input to the MC
    "mc_input": {
        "delay": 150.0,
        "dur": 5000.0,
        "amp": 4.0,  # nA
    },

    # MC -> GC excitation
    "mc_to_gc": {
        "sections": [0, 1, 2],
        "include_soma": False,
        "gmax": 100.0,
        "weight": 1.0,
        "delay": 1.0,
        "threshold": 0.0,
    },

    # GC NMDA control
    "block_gc_nmda": False,
    # nmdafactor used by AmpaNmdaSyn on the GC side; keep boolean + factor so
    # it's easy to switch off NMDA completely.
    "gc_nmda_factor": 0.0,

    # GC -> MC inhibition
    "gc_to_mc": {
        "gmax": 0.2,
        "weight": 10.0,
        "delay": 1.0,
        "threshold": -40.0,
        "tau1": 1.0,
        "tau2": 50.0,
    },
}

# Derive dependent values and export top-level names for compatibility
tstop = params["tstop"]
mc_input_delay = params["mc_input"]["delay"]
mc_input_dur = params["mc_input"]["dur"]
mc_input_amp = params["mc_input"]["amp"]
mc_to_gc_sections = params["mc_to_gc"]["sections"]
mc_to_gc_include_soma = params["mc_to_gc"]["include_soma"]
mc_to_gc_gmax = params["mc_to_gc"]["gmax"]
mc_to_gc_weight = params["mc_to_gc"]["weight"]
mc_to_gc_delay = params["mc_to_gc"]["delay"]
mc_to_gc_threshold = params["mc_to_gc"]["threshold"]
block_gc_nmda = params["block_gc_nmda"]
# If the block flag is True, force factor to 0.0
gc_nmda_factor = 0.0 if block_gc_nmda else params["gc_nmda_factor"]

gc_to_mc_gmax = params["gc_to_mc"]["gmax"]
gc_to_mc_weight = params["gc_to_mc"]["weight"]
gc_to_mc_delay = params["gc_to_mc"]["delay"]
gc_to_mc_threshold = params["gc_to_mc"]["threshold"]
gc_to_mc_tau1 = params["gc_to_mc"]["tau1"]
gc_to_mc_tau2 = params["gc_to_mc"]["tau2"]

# Expose params for interactive editing
params


In [ ]:
import copy

PARAM_PRESETS = {
    "base": copy.deepcopy(params),
    "official": {
        "tstop": 300.0,
        "mc_input": {
            "delay": 50.0,
            "dur": 150.0,
            "amp": 1.5,
        },
        "mc_to_gc": {
            "sections": [0],
            "include_soma": False,
            "gmax": 0.1,
            "weight": 1.0,
            "delay": 0.5,
            "threshold": 0.0,
        },
        "block_gc_nmda": False,
        "gc_nmda_factor": 0.0035,
        "gc_to_mc": {
            "gmax": 0.005,
            "weight": 1.0,
            "delay": 0.5,
            "threshold": 0.0,
            "tau1": 1.0,
            "tau2": 100.0,
        },
    },
}

def _sync_top_level_params():
    global tstop, mc_input_delay, mc_input_dur, mc_input_amp
    global mc_to_gc_sections, mc_to_gc_include_soma, mc_to_gc_gmax
    global mc_to_gc_weight, mc_to_gc_delay, mc_to_gc_threshold
    global block_gc_nmda, gc_nmda_factor
    global gc_to_mc_gmax, gc_to_mc_weight, gc_to_mc_delay
    global gc_to_mc_threshold, gc_to_mc_tau1, gc_to_mc_tau2

    tstop = params["tstop"]
    mc_input_delay = params["mc_input"]["delay"]
    mc_input_dur = params["mc_input"]["dur"]
    mc_input_amp = params["mc_input"]["amp"]
    mc_to_gc_sections = params["mc_to_gc"]["sections"]
    mc_to_gc_include_soma = params["mc_to_gc"]["include_soma"]
    mc_to_gc_gmax = params["mc_to_gc"]["gmax"]
    mc_to_gc_weight = params["mc_to_gc"]["weight"]
    mc_to_gc_delay = params["mc_to_gc"]["delay"]
    mc_to_gc_threshold = params["mc_to_gc"]["threshold"]
    block_gc_nmda = params["block_gc_nmda"]
    gc_nmda_factor = 0.0 if block_gc_nmda else params["gc_nmda_factor"]
    gc_to_mc_gmax = params["gc_to_mc"]["gmax"]
    gc_to_mc_weight = params["gc_to_mc"]["weight"]
    gc_to_mc_delay = params["gc_to_mc"]["delay"]
    gc_to_mc_threshold = params["gc_to_mc"]["threshold"]
    gc_to_mc_tau1 = params["gc_to_mc"]["tau1"]
    gc_to_mc_tau2 = params["gc_to_mc"]["tau2"]

def _deep_update(target, updates):
    for key, value in updates.items():
        if isinstance(value, dict) and isinstance(target.get(key), dict):
            _deep_update(target[key], value)
        else:
            target[key] = value
    return target

def apply_params(use_preset=None, overrides=None):
    if use_preset is None:
        updated = copy.deepcopy(PARAM_PRESETS["base"])
    else:
        if use_preset not in PARAM_PRESETS:
            raise KeyError(f"Unknown preset: {use_preset}")
        updated = copy.deepcopy(PARAM_PRESETS[use_preset])

    if overrides:
        _deep_update(updated, overrides)

    params.clear()
    params.update(updated)
    _sync_top_level_params()
    return params

def setup_two_cell_circuit():
    h('forall delete_section()')

    mc = MC1()
    gc = GC1()

    mc_stim = h.IClamp(mc.soma(0.5))
    mc_stim.delay = mc_input_delay
    mc_stim.dur = mc_input_dur
    mc_stim.amp = mc_input_amp

    gc_target_sections = [gc.cell.apic[i] for i in mc_to_gc_sections]
    if mc_to_gc_include_soma:
        gc_target_sections.append(gc.soma)

    mc_to_gc_syns = []
    mc_to_gc_netcons = []
    for section in gc_target_sections:
        syn = h.AmpaNmdaSyn(section(0.5))
        syn.gmax = mc_to_gc_gmax
        syn.nmdafactor = gc_nmda_factor

        nc = h.NetCon(mc.soma(0.5)._ref_v, syn, sec=mc.soma)
        nc.threshold = mc_to_gc_threshold
        nc.delay = mc_to_gc_delay
        nc.weight[0] = mc_to_gc_weight

        mc_to_gc_syns.append(syn)
        mc_to_gc_netcons.append(nc)

    gc_to_mc_syn = h.GabaSyn(mc.cell.dend[0](0.5))
    gc_to_mc_syn.gmax = gc_to_mc_gmax
    gc_to_mc_syn.tau1 = gc_to_mc_tau1
    gc_to_mc_syn.tau2 = gc_to_mc_tau2

    gc_to_mc_netcon = h.NetCon(gc.soma(0.5)._ref_v, gc_to_mc_syn, sec=gc.soma)
    gc_to_mc_netcon.threshold = gc_to_mc_threshold
    gc_to_mc_netcon.delay = gc_to_mc_delay
    gc_to_mc_netcon.weight[0] = gc_to_mc_weight

    return {
        "mc": mc,
        "gc": gc,
        "mc_stim": mc_stim,
        "mc_to_gc_syns": mc_to_gc_syns,
        "mc_to_gc_netcons": mc_to_gc_netcons,
        "gc_to_mc_syn": gc_to_mc_syn,
        "gc_to_mc_netcon": gc_to_mc_netcon,
    }

_sync_top_level_params()

In [ ]:
# Setup two-cell circuit using centralized helper
circuit_objs = setup_two_cell_circuit()

# expose objects for the rest of the notebook
mc = circuit_objs["mc"]
gc = circuit_objs["gc"]
mc_stim = circuit_objs["mc_stim"]
mc_to_gc_syns = circuit_objs["mc_to_gc_syns"]
mc_to_gc_netcons = circuit_objs["mc_to_gc_netcons"]
gc_to_mc_syn = circuit_objs["gc_to_mc_syn"]
gc_to_mc_netcon = circuit_objs["gc_to_mc_netcon"]

# Print a quick summary
print(f"Created MC and GC cells. MC input amp={mc_stim.amp} nA, delay={mc_stim.delay} ms")
print(f"MC->GC: {len(mc_to_gc_syns)} synapses, GC->MC gmax={gc_to_mc_syn.gmax}")

In [ ]:
# Legacy inline setup removed.
# Use the centralized helper cell above so the notebook builds the two-cell circuit in one place only.

In [ ]:
# Record MC and GC outputs
time = h.Vector().record(h._ref_t)
mc_voltage = h.Vector().record(mc.soma(0.5)._ref_v)
gc_voltage = h.Vector().record(gc.soma(0.5)._ref_v)

# Record the external MC input
mc_input_current = h.Vector().record(mc_stim._ref_i)

# Record GC excitatory input from every MC -> GC synapse
gc_input_total_vecs = [h.Vector().record(syn._ref_i) for syn in mc_to_gc_syns]
gc_input_ampa_vecs = [h.Vector().record(syn._ref_iampa) for syn in mc_to_gc_syns]
gc_input_nmda_vecs = [h.Vector().record(syn._ref_inmda) for syn in mc_to_gc_syns]

# Record GC inhibitory output onto the MC
gc_output_current = h.Vector().record(gc_to_mc_syn._ref_i)
gc_output_conductance = h.Vector().record(gc_to_mc_syn._ref_g)

In [ ]:
h.finitialize(-65.0)
h.continuerun(tstop)

t = np.array(time)
mc_v = np.array(mc_voltage)
gc_v = np.array(gc_voltage)
mc_i = np.array(mc_input_current)

gc_input_total = np.sum([np.array(v) for v in gc_input_total_vecs], axis=0)
gc_input_ampa = np.sum([np.array(v) for v in gc_input_ampa_vecs], axis=0)
gc_input_nmda = np.sum([np.array(v) for v in gc_input_nmda_vecs], axis=0)

gc_out_i = np.array(gc_output_current)
gc_out_g = np.array(gc_output_conductance)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axes[0].plot(t, mc_i, color="black")
axes[0].set_ylabel("nA")
axes[0].set_title("MC External Input")

axes[1].plot(t, gc_input_total, label="Total", color="tab:blue")
axes[1].plot(t, gc_input_ampa, label="AMPA", color="tab:orange", alpha=0.8)
axes[1].plot(t, gc_input_nmda, label="NMDA", color="tab:green", alpha=0.8)
axes[1].set_ylabel("nA")
axes[1].set_title("GC Excitatory Input From MC")
axes[1].legend(loc="best")

axes[2].plot(t, gc_out_i, label="GC -> MC inhibitory current", color="tab:red")
axes[2].plot(t, gc_out_g, label="GC -> MC conductance", color="tab:purple", alpha=0.8)
axes[2].set_ylabel("nA / uS")
axes[2].set_title("GC Inhibitory Output Onto MC")
axes[2].legend(loc="best")

axes[3].plot(t, mc_v, label="MC soma", color="tab:blue")
axes[3].plot(t, gc_v, label="GC soma", color="tab:red")
axes[3].set_xlabel("Time (ms)")
axes[3].set_ylabel("mV")
axes[3].set_title("Cell Outputs")
axes[3].legend(loc="best")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import spectrogram

In [ ]:
from scipy.signal import butter, filtfilt, spectrogram
from scipy.interpolate import interp1d
from scipy.signal import spectrogram
import matplotlib.pyplot as plt

# original nonuniform data
t_ms = np.array(t)
signal = np.array(mc_v)

# choose a uniform grid in ms
dt_uniform_ms = 0.01
t_uniform_ms = np.arange(t_ms[0], t_ms[-1], dt_uniform_ms)

# interpolate signal onto the uniform grid
f = interp1d(t_ms, signal, kind="linear", fill_value="extrapolate")
signal_uniform = f(t_uniform_ms)

# sampling frequency in Hz
fs = 1000.0 / dt_uniform_ms

b, a = butter(3, 100 / (fs / 2), btype="low")
mc_v_smooth = filtfilt(b, a, signal_uniform)

freqs, times, power = spectrogram(
  mc_v_smooth,
  fs=fs,
  nperseg=20000,
  noverlap=500,
  scaling="density",
  mode="psd",
)

plt.figure(figsize=(10, 4))
plt.pcolormesh(times * 1000, freqs, 10 * np.log10(power + 1e-4), shading="auto")
plt.xlabel("Time (ms)")
plt.ylabel("Frequency (Hz)")
plt.ylim(0, 200)
plt.colorbar(label="Power (dB)")
plt.tight_layout()
plt.show()


In [ ]:
freqs

In [ ]:
print(f"MC peak voltage: {mc_v.max():.2f} mV")
print(f"GC peak voltage: {gc_v.max():.2f} mV")
print(f"GC peak excitatory input current: {gc_input_total.min():.4f} nA")
print(f"GC peak inhibitory output conductance: {gc_out_g.max():.4f} uS")
print(f"GC NMDA blocked: {block_gc_nmda}")

In [ ]:
# Official reciprocal-connection values from the slice JSON / mechanism defaults
# (Replaced by params["presets"]["official"]; kept this cell for compatibility but
# now it simply applies the official preset.)

# Apply the official preset to override base params for this demo
apply_params(use_preset="official")

# Export the official-time names for any downstream code that referenced them
official_tstop = tstop
official_mc_input_delay = mc_input_delay
official_mc_input_dur = mc_input_dur
official_mc_input_amp = mc_input_amp

official_mc_to_gc_sections = mc_to_gc_sections
official_mc_to_gc_include_soma = mc_to_gc_include_soma
official_mc_to_gc_gmax = mc_to_gc_gmax
official_mc_to_gc_weight = mc_to_gc_weight
official_mc_to_gc_delay = mc_to_gc_delay
official_mc_to_gc_threshold = mc_to_gc_threshold

official_block_gc_nmda = block_gc_nmda
official_gc_nmda_factor = gc_nmda_factor

official_gc_to_mc_gmax = gc_to_mc_gmax
official_gc_to_mc_weight = gc_to_mc_weight
official_gc_to_mc_delay = gc_to_mc_delay
official_gc_to_mc_threshold = gc_to_mc_threshold
official_gc_to_mc_tau1 = gc_to_mc_tau1
official_gc_to_mc_tau2 = gc_to_mc_tau2

# Show the applied official settings
_applied_official = {
    "tstop": official_tstop,
    "mc_input_delay": official_mc_input_delay,
    "mc_input_dur": official_mc_input_dur,
    "mc_input_amp": official_mc_input_amp,
}
_applied_official

In [ ]:
# Legacy explicit official-value dump removed.
# The preset application cell above now owns those values via apply_params(use_preset="official").

In [ ]:
official_mc = MC1()
official_gc = GC1()

official_mc_stim = h.IClamp(official_mc.soma(0.5))
official_mc_stim.delay = official_mc_input_delay
official_mc_stim.dur = official_mc_input_dur
official_mc_stim.amp = official_mc_input_amp

official_gc_target_sections = [official_gc.cell.apic[i] for i in official_mc_to_gc_sections]
if official_mc_to_gc_include_soma:
    official_gc_target_sections.append(official_gc.soma)

official_mc_to_gc_syns = []
official_mc_to_gc_netcons = []
for sec in official_gc_target_sections:
    syn = h.AmpaNmdaSyn(sec(0.5))
    syn.gmax = official_mc_to_gc_gmax
    syn.nmdafactor = official_gc_nmda_factor

    nc = h.NetCon(official_mc.soma(0.5)._ref_v, syn, sec=official_mc.soma)
    nc.threshold = official_mc_to_gc_threshold
    nc.delay = official_mc_to_gc_delay
    nc.weight[0] = official_mc_to_gc_weight

    official_mc_to_gc_syns.append(syn)
    official_mc_to_gc_netcons.append(nc)

official_gc_to_mc_syn = h.GabaSyn(official_mc.cell.dend[0](0.5))
official_gc_to_mc_syn.gmax = official_gc_to_mc_gmax
official_gc_to_mc_syn.tau1 = official_gc_to_mc_tau1
official_gc_to_mc_syn.tau2 = official_gc_to_mc_tau2

official_gc_to_mc_netcon = h.NetCon(official_gc.soma(0.5)._ref_v, official_gc_to_mc_syn, sec=official_gc.soma)
official_gc_to_mc_netcon.threshold = official_gc_to_mc_threshold
official_gc_to_mc_netcon.delay = official_gc_to_mc_delay
official_gc_to_mc_netcon.weight[0] = official_gc_to_mc_weight

official_time = h.Vector().record(h._ref_t)
official_mc_voltage = h.Vector().record(official_mc.soma(0.5)._ref_v)
official_gc_voltage = h.Vector().record(official_gc.soma(0.5)._ref_v)
official_mc_input_current = h.Vector().record(official_mc_stim._ref_i)

official_gc_input_total_vecs = [h.Vector().record(syn._ref_i) for syn in official_mc_to_gc_syns]
official_gc_input_ampa_vecs = [h.Vector().record(syn._ref_iampa) for syn in official_mc_to_gc_syns]
official_gc_input_nmda_vecs = [h.Vector().record(syn._ref_inmda) for syn in official_mc_to_gc_syns]
official_gc_output_current = h.Vector().record(official_gc_to_mc_syn._ref_i)
official_gc_output_conductance = h.Vector().record(official_gc_to_mc_syn._ref_g)

h.finitialize(-65.0)
h.continuerun(official_tstop)

official_t = np.array(official_time)
official_mc_v = np.array(official_mc_voltage)
official_gc_v = np.array(official_gc_voltage)
official_mc_i = np.array(official_mc_input_current)

official_gc_input_total = np.sum([np.array(v) for v in official_gc_input_total_vecs], axis=0)
official_gc_input_ampa = np.sum([np.array(v) for v in official_gc_input_ampa_vecs], axis=0)
official_gc_input_nmda = np.sum([np.array(v) for v in official_gc_input_nmda_vecs], axis=0)

official_gc_out_i = np.array(official_gc_output_current)
official_gc_out_g = np.array(official_gc_output_conductance)

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12, 12), sharex=True)

axes[0].plot(official_t, official_mc_i, color="black")
axes[0].set_ylabel("nA")
axes[0].set_title("Official-Value Demo: MC External Input")

axes[1].plot(official_t, official_gc_input_total, label="Total", color="tab:blue")
axes[1].plot(official_t, official_gc_input_ampa, label="AMPA", color="tab:orange", alpha=0.8)
axes[1].plot(official_t, official_gc_input_nmda, label="NMDA", color="tab:green", alpha=0.8)
axes[1].set_ylabel("nA")
axes[1].set_title("Official-Value Demo: GC Excitatory Input From MC")
axes[1].legend(loc="best")

axes[2].plot(official_t, official_gc_out_i, label="GC -> MC inhibitory current", color="tab:red")
axes[2].plot(official_t, official_gc_out_g, label="GC -> MC conductance", color="tab:purple", alpha=0.8)
axes[2].set_ylabel("nA / uS")
axes[2].set_title("Official-Value Demo: GC Inhibitory Output Onto MC")
axes[2].legend(loc="best")

axes[3].plot(official_t, official_mc_v, label="MC soma", color="tab:blue")
axes[3].plot(official_t, official_gc_v, label="GC soma", color="tab:red")
axes[3].set_xlabel("Time (ms)")
axes[3].set_ylabel("mV")
axes[3].set_title("Official-Value Demo: Cell Outputs")
axes[3].legend(loc="best")

plt.tight_layout()
plt.show()

In [ ]:
print(f"Official MC peak voltage: {official_mc_v.max():.2f} mV")
print(f"Official GC peak voltage: {official_gc_v.max():.2f} mV")
print(f"Official GC peak excitatory input current: {official_gc_input_total.min():.6f} nA")
print(f"Official GC peak inhibitory output conductance: {official_gc_out_g.max():.6f} uS")
print(f"Official GC NMDA blocked: {official_block_gc_nmda}")

## Parameter Sweep Animations

This section runs a single-parameter sweep by launching each NEURON simulation in a fresh subprocess. That avoids the kernel crashes you were seeing when rebuilding HOC objects repeatedly inside one notebook session.

The helper also forces fixed-step integration (`cvode` off, explicit `dt`) so the spectrogram frames use uniformly sampled traces.

In [ ]:
import numpy as np
from IPython.display import HTML
from mc_gc_sweep import (
    PRESETS,
    run_parameter_sweep,
    animate_trace_sweep,
    animate_spectrogram_sweep,
)

In [ ]:
# Sweep one parameter at a time.
# The base sweep config now inherits the first simulation values from above,
# and only the swept parameter changes from frame to frame.
sorted(PRESETS["tuned"].keys())

SWEEP_PRESET = "tuned"  # structural defaults; overridden below with the first-simulation values
SWEEP_PARAMETER = "mc_input_amp"
SWEEP_VALUES = np.linspace(1.0, 4.0, 15)

# Shared overrides applied to every run in the sweep.
# These reuse the first-simulation settings defined earlier in the notebook.
SWEEP_OVERRIDES = {
    "tstop": tstop,
    "dt_ms": 0.5,
    "init_v": -65.0,
    "mc_input_delay": mc_input_delay,
    "mc_input_dur": mc_input_dur,
    "mc_input_amp": mc_input_amp,
    "mc_to_gc_sections": mc_to_gc_sections,
    "mc_to_gc_include_soma": mc_to_gc_include_soma,
    "mc_to_gc_gmax": mc_to_gc_gmax,
    "mc_to_gc_weight": mc_to_gc_weight,
    "mc_to_gc_delay": mc_to_gc_delay,
    "mc_to_gc_threshold": mc_to_gc_threshold,
    "gc_nmda_factor": gc_nmda_factor,
    "gc_to_mc_gmax": gc_to_mc_gmax,
    "gc_to_mc_weight": gc_to_mc_weight,
    "gc_to_mc_delay": gc_to_mc_delay,
    "gc_to_mc_threshold": gc_to_mc_threshold,
    "gc_to_mc_tau1": gc_to_mc_tau1,
    "gc_to_mc_tau2": gc_to_mc_tau2,
    "block_gc_nmda": block_gc_nmda,
}

# Signal to use for the animated spectrogram.
# Good starting points: "mc_v", "gc_v", "gc_input_total", "gc_out_i", "gc_out_g"
SPECTROGRAM_SIGNAL = "gc_out_g"
SPECTROGRAM_MAX_FREQ = 100
SPECTROGRAM_NPERSEG = 1024
SPECTROGRAM_NOVERLAP = 960
ANIMATION_INTERVAL_MS = 700

In [ ]:
sweep_results = run_parameter_sweep(
    SWEEP_PARAMETER,
    SWEEP_VALUES,
    preset=SWEEP_PRESET,
    overrides=SWEEP_OVERRIDES,
)

for result in sweep_results:
    value = result["swept_value"]
    print(
        f"{SWEEP_PARAMETER}={value:.4g} | "
        f"MC peak={result['mc_v'].max():.2f} mV | "
        f"GC peak={result['gc_v'].max():.2f} mV | "
        f"GC->MC gmax trace peak={result['gc_out_g'].max():.6f} uS"
    )

In [ ]:
trace_anim = animate_trace_sweep(
    sweep_results,
    parameter_name=SWEEP_PARAMETER,
    interval=ANIMATION_INTERVAL_MS,
)
HTML(trace_anim.to_jshtml())

In [ ]:
spectrogram_anim = animate_spectrogram_sweep(
    sweep_results,
    signal_key=SPECTROGRAM_SIGNAL,
    parameter_name=SWEEP_PARAMETER,
    max_freq=SPECTROGRAM_MAX_FREQ,
    nperseg=SPECTROGRAM_NPERSEG,
    noverlap=SPECTROGRAM_NOVERLAP,
    interval=ANIMATION_INTERVAL_MS,
)
HTML(spectrogram_anim.to_jshtml())